# Домашнее задание 1 - 15 баллов
1. Загрузите набор данных lenta-ru-news с помощью библиотеки Corus для задачи классификации текстов по топикам (пригодятся атрибуты title, text, topic)
2. Подготовьте данные к обучению: - 3 балла
3. Выделите репрезентативную часть датасета размеро 100_000 текстов
4. Предобработайте данные: реализуйте оптимальную, на ваш взгляд, предобработку текстов (нормализация, очистка, стемминг/лемматизация и т.п.) и таргета. Постарайтесь добиться приемлемой скорости обработки за счет оптимизаций.
5. Кратко опишите пайплайн предобработки, на котором остановились, и почему.
6. Разделите датасет на обучающую, валидационную и тестовую выборки со стратификацией в пропорции 60/20/20. В качестве целевой переменной используйте атрибут topic
7. Замерьте базовое качество с любым dummy-бейзлайном - 1 балл
8. Обучите модель sklearn.linear_model.LogisticRegression с двумя вариантами векторизации: 3 балла
sklearn.feature_extraction.text.CountVectorizer
sklearn.feature_extraction.text.TfidfVectorizer
9. Попробуйте улучшить качество, подобрав оптимальные гиперпараметры трансформаций и модели на кросс-валидации 2 балла
10. Оцените качество лучшего пайплайна на отложенной выборке, проведите анализ ошибок модели, сделайте выводы - 2 балла

In [ ]:
!wget https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz

## Загрузка данных

Загрузим 'lenta-ru-news' датасет


In [ ]:
print("Installing corus library...")
!pip install corus
print("Corus library installed.")

Installing corus library...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.7/83.7 kB 5.0 MB/s eta 0:00:00
Corus library installed.


In [ ]:
# Загрузка датасета lenta-ru-news с помощью библиотеки corus.
# corus предоставляет удобный генератор записей из сжатого CSV,
# что позволяет не загружать весь файл в память сразу.
from corus import load_lenta
path = 'lenta-ru-news.csv.gz'
records = load_lenta(path)


In [ ]:
next(records)

LentaRecord(
    url='https://lenta.ru/news/2018/12/14/cancer/',
    title='Названы регионы России с\xa0самой высокой смертностью от\xa0рака',
    text='Вице-премьер по социальным вопросам Татьяна Голикова рассказала, в каких регионах России зафиксирована наиболее высокая смертность от рака, сообщает РИА Новости. По словам Голиковой, чаще всего онкологические заболевания становились причиной смерти в Псковской, Тверской, Тульской и Орловской областях, а также в Севастополе. Вице-премьер напомнила, что главные факторы смертности в России — рак и болезни системы кровообращения. В начале года стало известно, что смертность от онкологических заболеваний среди россиян снизилась впервые за три года. По данным Росстата, в 2017 году от рака умерли 289 тысяч человек. Это на 3,5 процента меньше, чем годом ранее.',
    topic='Россия',
    tags='Общество',
    date=None
)

## Преобразуем данные в формат pandas df



In [ ]:
import pandas as pd

# Итерируем по генератору и собираем нужные поля: title, text, topic.
# Используем только три атрибута из всех доступных (url, tags, date не нужны).
data = []
for record in records:
    data.append({
        'title': record.title,
        'text': record.text,
        'topic': record.topic
    })

df = pd.DataFrame(data)
print("DataFrame created successfully.")
df.head()

DataFrame created successfully.


,title,text,topic
0,Австрия не представила доказательств вины росс...,Австрийские правоохранительные органы не предс...,Спорт
1,Обнаружено самое счастливое место на планете,Сотрудники социальной сети Instagram проанализ...,Путешествия
2,В США раскрыли сумму расходов на расследование...,С начала расследования российского вмешательст...,Мир
3,Хакеры рассказали о планах Великобритании зами...,Хакерская группировка Anonymous опубликовала н...,Мир
4,Архиепископ канонической УПЦ отказался прийти ...,Архиепископ канонической Украинской православн...,Бывший СССР


## Засемплируем часть данных (с фиксированным сидом) + выделим фичу full_text



In [ ]:
# Фиксируем random_state=42 для воспроизводимости выборки.
# Берём 100 000 записей из общего датасета (~800k записей).
# full_text = title + text: заголовок часто содержит ключевые слова,
# поэтому объединение заголовка и тела статьи улучшает качество классификации.
df_sampled = df.sample(n=100000, random_state=42).reset_index(drop=True)
df_sampled['full_text'] = df_sampled['title'] + ' ' + df_sampled['text']

print("Sampled DataFrame created with 100,000 rows and 'full_text' column.")
df_sampled.head()

Sampled DataFrame created with 100,000 rows and 'full_text' column.


,title,text,topic,full_text
0,Гнилые помидоры оказались источником электроэн...,Группа ученых из Принстонского университета и ...,Наука и техника,Гнилые помидоры оказались источником электроэн...
1,Двое дагестанских боевиков осуждены за убийств...,В Северо-Кавказском окружном военном суде выне...,Силовые структуры,Двое дагестанских боевиков осуждены за убийств...
2,Путин поздравил российских иудеев с праздником...,Президент России Владимир Путин поздравил евре...,Россия,Путин поздравил российских иудеев с праздником...
3,Женская грудь стала модным культом,Американская супермодель и участница реалити-ш...,Ценности,Женская грудь стала модным культом Американска...
4,Шотландец грыз ногти и заработал заражение крови,Житель британского города Уэстклифф-он-Си (гра...,Из жизни,Шотландец грыз ногти и заработал заражение кро...


In [ ]:
# установим нлтк
import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

# Используем SnowballStemmer для русского языка — быстрее, чем лемматизация (PyMorphy2),
# но менее точный: 'играть'/'игра' -> 'игр'. Для задачи классификации по темам
# стемминг даёт приемлемое качество при высокой скорости обработки 100k текстов.
stemmer = SnowballStemmer('russian')
russian_stopwords = stopwords.words('russian')

def preprocess_text(text):
    text = text.lower()
    # удаляем пунктуацию и спецсимволы, оставляем только буквы
    text = re.sub(r'[^a-zA-Zа-яА-Я\s]', '', text)
    words = re.findall(r'\b\w+\b', text)
    # фильтруем стоп-слова и короткие токены, затем применяем стемминг
    words = [stemmer.stem(word) for word in words if word not in russian_stopwords and len(word) > 2]
    # обычно лемматизация лучше работает, но решил взять стемминг
    return ' '.join(words)

df_sampled['cleaned_text'] = df_sampled['full_text'].apply(preprocess_text)

print("Text preprocessing complete. Displaying first 5 cleaned texts:")
print(df_sampled['cleaned_text'].head().tolist())

Text preprocessing complete. Displaying first 5 cleaned texts:
['гнил помидор оказа источник электроэнерг групп учен принстонск университет университет флорид галф кост реш использова испорчен томат качеств источник энерг результат работ исследовател представ ежегодн собран американск химическ обществ аспирантк намит шрест котор работа проект утвержда томат могут подходя основ химическ источник ток бактер окисля мякот плод высвобожда электрон созда ток особ рол процесс игра ликопин пигмент прида томат красн цвет стимулир генерац электрическ заряд учен утвержда энергетическ эффективн микробн разложен томат превосход чист химическ веществ котор обычн использ гальваническ элемент хот выходн мощност устройств мал миллиграмм томатн отход создаст лиш ватт электроэнерг исследовател обеща показател увелич нескольк порядк расчет автор исследован использова испорчен томат котор набира штат флорид год энерг хват дне работ аттракцион парк disney world органическ промышлен отход широк использ качес

In [ ]:
print(1)

1


In [ ]:
df_sampled

,title,text,topic,full_text,cleaned_text
0,Гнилые помидоры оказались источником электроэн...,Группа ученых из Принстонского университета и ...,Наука и техника,Гнилые помидоры оказались источником электроэн...,гнил помидор оказа источник электроэнерг групп...
1,Двое дагестанских боевиков осуждены за убийств...,В Северо-Кавказском окружном военном суде выне...,Силовые структуры,Двое дагестанских боевиков осуждены за убийств...,дво дагестанск боевик осужд убийств начальник ...
2,Путин поздравил российских иудеев с праздником...,Президент России Владимир Путин поздравил евре...,Россия,Путин поздравил российских иудеев с праздником...,путин поздрав российск иуде праздник пес прези...
3,Женская грудь стала модным культом,Американская супермодель и участница реалити-ш...,Ценности,Женская грудь стала модным культом Американска...,женск груд стал модн культ американск супермод...
4,Шотландец грыз ногти и заработал заражение крови,Житель британского города Уэстклифф-он-Си (гра...,Из жизни,Шотландец грыз ногти и заработал заражение кро...,шотландец грыз ногт заработа заражен кров жите...
...,...,...,...,...,...
99995,Якуб Дениев освобожден из плена бесплатно,В результате проведения специальной операции с...,Россия,Якуб Дениев освобожден из плена бесплатно В ре...,якуб ден освобожд плен бесплатн результат пров...
99996,При крушении российского вертолета у берегов Н...,Восемь человек погибли при падении российского...,Мир,При крушении российского вертолета у берегов Н...,крушен российск вертолет берег норвег погибл в...
99997,Китай и страны Латинской Америки заинтересовал...,Управляющий директор Улан-Удэнского авиационно...,Силовые структуры,Китай и страны Латинской Америки заинтересовал...,кита стран латинск америк заинтересова арктиче...
99998,Дэвид Бекхэм подарил себе на 40-летие аккаунт ...,Известный футболист Дэвид Бекхэм зарегистриров...,Спорт,Дэвид Бекхэм подарил себе на 40-летие аккаунт ...,дэвид бекхэм подар лет аккаунт instagram извес...


In [ ]:
print("Distribution of topics in the sampled dataset:")
print(df_sampled['topic'].value_counts())

# необходимо подумать что делать с малопредставленными классами


Distribution of topics in the sampled dataset:
topic
Россия               21789
Мир                  18521
Экономика            10688
Спорт                 8735
Культура              7313
Бывший СССР           7204
Наука и техника       7076
Интернет и СМИ        6149
Из жизни              3754
Дом                   2837
Силовые структуры     2678
Бизнес                1030
Ценности               993
Путешествия            865
69-я параллель         171
Крым                   102
Культпросвет            48
                        21
Легпром                 15
Библиотека               9
МедНовости               1
ЧМ-2014                  1
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# уберем странный пустой класс и классы с 1 примером
df_sampled = df_sampled[~df_sampled["topic"].isin(["", "МедНовости", "ЧМ-2014"])]


In [ ]:
df_sampled.head(100)

,title,text,topic,full_text,cleaned_text
0,Гнилые помидоры оказались источником электроэн...,Группа ученых из Принстонского университета и ...,Наука и техника,Гнилые помидоры оказались источником электроэн...,гнил помидор оказа источник электроэнерг групп...
1,Двое дагестанских боевиков осуждены за убийств...,В Северо-Кавказском окружном военном суде выне...,Силовые структуры,Двое дагестанских боевиков осуждены за убийств...,дво дагестанск боевик осужд убийств начальник ...
2,Путин поздравил российских иудеев с праздником...,Президент России Владимир Путин поздравил евре...,Россия,Путин поздравил российских иудеев с праздником...,путин поздрав российск иуде праздник пес прези...
3,Женская грудь стала модным культом,Американская супермодель и участница реалити-ш...,Ценности,Женская грудь стала модным культом Американска...,женск груд стал модн культ американск супермод...
4,Шотландец грыз ногти и заработал заражение крови,Житель британского города Уэстклифф-он-Си (гра...,Из жизни,Шотландец грыз ногти и заработал заражение кро...,шотландец грыз ногт заработа заражен кров жите...
...,...,...,...,...,...
95,В Иране при взрыве погиб физик-ядерщик,Иранский ученый-ядерщик Массуд Али Мохаммади (...,Мир,В Иране при взрыве погиб физик-ядерщик Ирански...,иран взрыв погиб физикядерщик иранск ученыйяде...
96,Ростовский ОМОН разнял бывших спецназовцев и к...,Сотрудники правоохранительных органов Ростова-...,Россия,Ростовский ОМОН разнял бывших спецназовцев и к...,ростовск омон разня бывш спецназовц кавказц со...
97,На МКС починили систему по производству кислорода,"Космонавты, работающие на Международной космич...",Наука и техника,На МКС починили систему по производству кислор...,мкс почин сист производств кислород космонавт ...
98,Задержанная в Южной Осетии россиянка объявила ...,В цхинвальском СИЗО объявила голодовку российс...,Бывший СССР,Задержанная в Южной Осетии россиянка объявила ...,задержа южн осет россиянк объяв голодовк цхинв...


In [ ]:
X = df_sampled['cleaned_text']
y = df_sampled['topic']

# Сплит 60/20/20 со стратификацией: важно сохранить пропорции классов
# в каждой выборке, т.к. датасет несбалансирован (Россия ~22%, Легпром ~0.01%).
# Сначала отделяем 40% -> затем делим 50/50 на val и test.
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Data splitting complete. Shapes of the datasets:")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

Data splitting complete. Shapes of the datasets:
X_train shape: (59986,)
y_train shape: (59986,)
X_val shape: (19995,)
y_val shape: (19995,)
X_test shape: (19996,)
y_test shape: (19996,)


## Baseline Model



In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report

In [ ]:
dummy_clf = DummyClassifier(strategy='most_frequent')
dummy_clf.fit(X_train, y_train)
y_pred_dummy = dummy_clf.predict(X_test)

print("Classification Report for Dummy Classifier (most_frequent):")
print(classification_report(y_test, y_pred_dummy))

# baseline предсказывает плохо

Classification Report for Dummy Classifier (most_frequent):


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                   precision    recall  f1-score   support

   69-я параллель       0.00      0.00      0.00        34
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.00      0.00      0.00       206
      Бывший СССР       0.00      0.00      0.00      1441
              Дом       0.00      0.00      0.00       568
         Из жизни       0.00      0.00      0.00       751
   Интернет и СМИ       0.00      0.00      0.00      1230
             Крым       0.00      0.00      0.00        20
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.00      0.00      0.00      1463
          Легпром       0.00      0.00      0.00         3
              Мир       0.00      0.00      0.00      3704
  Наука и техника       0.00      0.00      0.00      1415
      Путешествия       0.00      0.00      0.00       173
           Россия       0.22      1.00      0.36      4358
Силовые структуры       0.00      0.00      0.00       

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Улучшим бейзлайн через CountVectorizer + LogisticRegression

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

In [ ]:
# CountVectorizer строит bag-of-words представление: каждое слово — отдельная
# размерность. Без ограничения max_features получаем ~171k признаков, что
# создаёт разреженную матрицу и замедляет обучение.
# fit только на train, чтобы избежать утечки данных (data leakage).
vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
X_val_vec = vectorizer.transform(X_val)

print("CountVectorizer fitted and data transformed.")
print(f"Shape of X_train_vec: {X_train_vec.shape}")
print(f"Shape of X_test_vec: {X_test_vec.shape}")
print(f"Shape of X_val_vec: {X_val_vec.shape}")

CountVectorizer fitted and data transformed.
Shape of X_train_vec: (59986, 171300)
Shape of X_test_vec: (19996, 171300)
Shape of X_val_vec: (19995, 171300)


In [ ]:
model_cv_lr = LogisticRegression(max_iter=1000, solver='saga', random_state=42)
model_cv_lr.fit(X_train_vec, y_train)

y_pred_cv_lr = model_cv_lr.predict(X_test_vec)

print("Classification Report for Logistic Regression with CountVectorizer:")
print(classification_report(y_test, y_pred_cv_lr))

# обучалась очень долго и качество нелучшее

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Classification Report for Logistic Regression with CountVectorizer:


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                   precision    recall  f1-score   support

   69-я параллель       0.83      0.44      0.58        34
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.45      0.32      0.37       206
      Бывший СССР       0.82      0.82      0.82      1441
              Дом       0.85      0.79      0.82       568
         Из жизни       0.61      0.58      0.60       751
   Интернет и СМИ       0.74      0.73      0.74      1230
             Крым       0.69      0.45      0.55        20
    Культпросвет        0.50      0.11      0.18         9
         Культура       0.87      0.87      0.87      1463
          Легпром       0.00      0.00      0.00         3
              Мир       0.78      0.81      0.79      3704
  Наука и техника       0.83      0.82      0.82      1415
      Путешествия       0.76      0.58      0.66       173
           Россия       0.77      0.82      0.79      4358
Силовые структуры       0.57      0.48      0.52       

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Возьмем линрег и TF-IDF


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# TF-IDF нормирует частоту слова на его встречаемость в корпусе (IDF),
# уменьшая вес слов-общеупотребительных (напр., 'сказал', 'стал') и
# усиливая вес редких, но информативных токенов.
# По сравнению с CountVectorizer даёт более качественные эмбеддинги
# для задач классификации при аналогичной вычислительной стоимости.
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("TfidfVectorizer fitted and data transformed.")
print(f"Shape of X_train_tfidf: {X_train_tfidf.shape}")
print(f"Shape of X_val_tfidf: {X_val_tfidf.shape}")
print(f"Shape of X_test_tfidf: {X_test_tfidf.shape}")

TfidfVectorizer fitted and data transformed.
Shape of X_train_tfidf: (59986, 171300)
Shape of X_val_tfidf: (19995, 171300)
Shape of X_test_tfidf: (19996, 171300)


In [ ]:
model_tfidf_lr = LogisticRegression(max_iter=1000, solver='saga', random_state=42)
model_tfidf_lr.fit(X_train_tfidf, y_train)

y_pred_tfidf_lr = model_tfidf_lr.predict(X_test_tfidf)

print("Classification Report for Logistic Regression with TfidfVectorizer:")
print(classification_report(y_test, y_pred_tfidf_lr, zero_division=0))

# Качество в среднем чуть лучше

Classification Report for Logistic Regression with TfidfVectorizer:
                   precision    recall  f1-score   support

   69-я параллель       1.00      0.06      0.11        34
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.68      0.11      0.19       206
      Бывший СССР       0.83      0.83      0.83      1441
              Дом       0.86      0.74      0.80       568
         Из жизни       0.70      0.54      0.61       751
   Интернет и СМИ       0.77      0.72      0.74      1230
             Крым       0.67      0.10      0.17        20
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.85      0.88      0.87      1463
          Легпром       0.00      0.00      0.00         3
              Мир       0.79      0.85      0.82      3704
  Наука и техника       0.81      0.84      0.83      1415
      Путешествия       0.81      0.46      0.59       173
           Россия       0.77      0.85      0.

## Hyperparameter Tuning (через GridSearchCV)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

Сделаем гридсерч на TF-IDF, так как он обучается в несколько раз быстрее, при этом качество сопоставимое



In [ ]:
pipeline_tfidf_lr = Pipeline([
    ('vect', TfidfVectorizer()),
    ('clf', LogisticRegression(random_state=42, solver='saga', max_iter=1000))
])

# Ищем оптимальные параметры векторизатора и классификатора одновременно:
# - max_features: ограничение словаря снижает размерность и ускоряет обучение
# - ngram_range=(1,2): биграммы могут улучшить качество (напр., 'северный кавказ')
# - C: обратная сила регуляризации; малое C -> сильная регуляризация
# - penalty: l1 даёт разреженные веса (feature selection), l2 обычно стабильнее
parameters_tfidf_lr = {
    'vect__max_features': [10000, 20000, 50000],
    'vect__ngram_range': [(1, 1), (1, 2)],
    'clf__C': [0.1, 1, 10],
    'clf__penalty': ['l1', 'l2']
}

# cv=3 для скорости; scoring='f1_weighted' учитывает дисбаланс классов
grid_search_tfidf_lr = GridSearchCV(pipeline_tfidf_lr, parameters_tfidf_lr, cv=3, n_jobs=-1, verbose=1, scoring='f1_weighted')

print("Fitting GridSearchCV for TfidfVectorizer and Logistic Regression...")
grid_search_tfidf_lr.fit(X_train, y_train)

print("Best parameters for TfidfVectorizer + Logistic Regression:", grid_search_tfidf_lr.best_params_)
print("Best f1_weighted score for TfidfVectorizer + Logistic Regression:", grid_search_tfidf_lr.best_score_)

Fitting GridSearchCV for TfidfVectorizer and Logistic Regression...
Fitting 3 folds for each of 36 candidates, totalling 108 fits


In [ ]:
# вынес отдельно best_params
best_params = {"vect__max_features": 20000, "vect__ngram_range": (1,2), "clf__C": 1, "clf__penalty": "l2"}

## Обучим пайплайн с лучшими гиперпараметрами


In [ ]:
best_pipeline = Pipeline([
    ('vect', TfidfVectorizer()),
    ('clf', LogisticRegression(random_state=42, solver='saga', max_iter=1000))
])

best_pipeline.set_params(**best_params)

print("Fitting best pipeline on training data...")
best_pipeline.fit(X_train, y_train)

y_pred_best = best_pipeline.predict(X_test)

print("Classification Report for Tuned TfidfVectorizer + Logistic Regression on Test Set:")
print(classification_report(y_test, y_pred_best, zero_division=0))

Fitting best pipeline on training data...
Classification Report for Tuned TfidfVectorizer + Logistic Regression on Test Set:
                   precision    recall  f1-score   support

   69-я параллель       1.00      0.12      0.21        34
       Библиотека       0.00      0.00      0.00         2
           Бизнес       0.76      0.17      0.27       206
      Бывший СССР       0.83      0.84      0.83      1441
              Дом       0.88      0.74      0.80       568
         Из жизни       0.69      0.55      0.61       751
   Интернет и СМИ       0.77      0.72      0.75      1230
             Крым       0.50      0.05      0.09        20
    Культпросвет        0.00      0.00      0.00         9
         Культура       0.86      0.87      0.87      1463
          Легпром       0.00      0.00      0.00         3
              Мир       0.79      0.85      0.81      3704
  Наука и техника       0.82      0.83      0.82      1415
      Путешествия       0.78      0.50      0.61

## Проанализируем ошибки



In [ ]:
misclassified_mask = (y_test != y_pred_best)

misclassified_df = pd.DataFrame({
    'full_text': df_sampled.loc[y_test.index, 'full_text'],
    'actual_topic': y_test,
    'predicted_topic': y_pred_best
})[misclassified_mask]

print("Misclassified samples DataFrame created. Displaying head:")
print(misclassified_df.head())
print(f"Total misclassified samples: {len(misclassified_df)}")

Misclassified samples DataFrame created. Displaying head:
                                               full_text    actual_topic  \
60750  Ходорковский дал онлайн-интервью в свой день р...  Интернет и СМИ   
35310  Минстрой задумался о лицензировании застройщик...             Дом   
43830  Элвис Пресли переквалифицировался в ангела-хра...        Из жизни   
74555  Фаст-фудом на стадионе "Спартак" займется "Шок...       Экономика   
76059  Университет Гамбурга будет бойкотировать рейти...             Мир   

      predicted_topic  
60750          Россия  
35310          Россия  
43830        Культура  
74555             Дом  
76059          Россия  
Total misclassified samples: 3760


In [ ]:
print("Distribution of actual topics in misclassified samples:")
print(misclassified_df['actual_topic'].value_counts())

print("\nDistribution of predicted topics in misclassified samples:")
print(misclassified_df['predicted_topic'].value_counts())

Distribution of actual topics in misclassified samples:
actual_topic
Россия               631
Мир                  572
Силовые структуры    352
Интернет и СМИ       343
Из жизни             341
Экономика            281
Наука и техника      235
Бывший СССР          224
Культура             189
Бизнес               172
Дом                  145
Путешествия           86
Спорт                 70
Ценности              56
69-я параллель        30
Крым                  19
Культпросвет           9
Легпром                3
Библиотека             2
Name: count, dtype: int64

Distribution of predicted topics in misclassified samples:
predicted_topic
Россия               1096
Мир                   855
Экономика             391
Наука и техника       266
Интернет и СМИ        263
Бывший СССР           257
Культура              207
Из жизни              183
Силовые структуры      77
Спорт                  65
Дом                    60
Путешествия            25
Бизнес                 11
Ценности        

### В основном плохо прогнозируем россию, мир (world), силовые структуры так как эти темы могут сильно пересекаться с другими похожими